# Tratamento Avançado de Dados e Dashboard de Modelos

Este notebook faz o pré-processamento completo e compara três modelos com validação cruzada e busca de hiperparâmetros. No final, há uma seção para gerar um dashboard usando Streamlit.

## 1. Importar bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

sns.set(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (11, 6)


## 2. Carregar os dados e identificar o dataset de treino

In [ ]:
train_path = "Data_Train.xlsx"
test_path = "Test_set.xlsx"
df_train = pd.read_excel(train_path)
df_test = pd.read_excel(test_path)
print("Treino:", df_train.shape)
print("Teste:", df_test.shape)
print("
Colunas treino:", df_train.columns.tolist())
print("Colunas teste:", df_test.columns.tolist())
print("
Alvo presente em treino?", "Price" in df_train.columns)
print("Alvo presente em teste?", "Price" in df_test.columns)


## 3. Inspeção inicial e EDA

In [ ]:
display(df_train.head())
display(df_train.info())
display(df_train.describe(include="all").T)


## 4. Definir funções de transformação de variáveis

In [ ]:
def extract_date_features(df, date_col="Date_of_Journey"):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors="coerce")
    df["journey_day"] = df[date_col].dt.day
    df["journey_month"] = df[date_col].dt.month
    df["journey_year"] = df[date_col].dt.year
    df.drop(columns=[date_col], inplace=True)
    return df

def duration_to_minutes(df, duration_col="Duration"):
    df = df.copy()
    durations = []
    for value in df[duration_col].astype(str):
        value = value.replace(" ", "")
        parts = value.split("h")
        hours = int(parts[0]) if parts[0] != "" else 0
        mins = int(parts[1].replace("m", "")) if len(parts) > 1 and parts[1] != "" else 0
        durations.append(hours * 60 + mins)
    df["duration_mins"] = durations
    df.drop(columns=[duration_col], inplace=True)
    return df

def stops_to_int(df, stops_col="Total_Stops"):
    df = df.copy()
    mapping = {
        "non-stop": 0,
        "1 stop": 1,
        "2 stops": 2,
        "3 stops": 3,
        "4 stops": 4
    }
    df["total_stops"] = df[stops_col].map(mapping)
    df.drop(columns=[stops_col], inplace=True)
    return df


## 5. Aplicar o tratamento de dados

In [ ]:
df = df_train.copy()
df = extract_date_features(df)
df = duration_to_minutes(df)
df = stops_to_int(df)

print("Nulos por coluna:")
print(df.isna().sum())
display(df.head())


## 6. Engenharia de features e pré-processamento

In [ ]:
features = [
    "Airline", "Source", "Destination", "Route", "Dep_Time", "Arrival_Time", "Additional_Info",
    "journey_day", "journey_month", "journey_year", "duration_mins", "total_stops"
]
target = "Price"
X = df[features]
y = df[target]

categorical_cols = [
    "Airline", "Source", "Destination", "Route", "Dep_Time", "Arrival_Time", "Additional_Info"
]
numeric_cols = ["journey_day", "journey_month", "journey_year", "duration_mins", "total_stops"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
])

X_preprocessed = preprocessor.fit_transform(X)
print("Formato após pré-processamento:", X_preprocessed.shape)


## 7. Separar treino e validação

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X_preprocessed, y, test_size=0.2, random_state=42)
print("Treino:", X_train.shape, "Validação:", X_valid.shape)


## 8. Modelos, busca de hiperparâmetros e validação cruzada

In [ ]:
models = {
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
    "KNN": KNeighborsRegressor(n_jobs=-1),
}

param_distributions = {
    "Random Forest": {
        "n_estimators": [100, 150, 200],
        "max_depth": [None, 10, 20],
        "min_samples_leaf": [1, 2, 4]
    },
    "XGBoost": {
        "n_estimators": [100, 150, 200],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.03, 0.06, 0.1],
        "subsample": [0.7, 0.85, 1.0]
    },
    "KNN": {
        "n_neighbors": [3, 5, 7, 9],
        "weights": ["uniform", "distance"]
    },
}

best_models = {}
search_results = []
for name, model in models.items():
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions[name],
        n_iter=8,
        scoring="neg_root_mean_squared_error",
        cv=3,
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    search_results.append({
        "Model": name,
        "Best Params": search.best_params_,
        "CV RMSE": -search.best_score_
    })
    print(f"{name}: melhor RMSE CV = {-search.best_score_:.2f}")
    print("  Melhor parâmetros:", search.best_params_)

results_search = pd.DataFrame(search_results).sort_values("CV RMSE")
results_search


## 9. Avaliação final no conjunto de validação

In [ ]:
eval_results = []
for name, model in best_models.items():
    preds = model.predict(X_valid)
    rmse = mean_squared_error(y_valid, preds, squared=False)
    r2 = r2_score(y_valid, preds)
    eval_results.append({
        "Model": name,
        "RMSE": rmse,
        "R2": r2
    })
    print(f"{name}: RMSE = {rmse:.2f}, R2 = {r2:.4f}")

results_eval = pd.DataFrame(eval_results).sort_values("RMSE")
results_eval


## 10. Gráficos de desempenho

In [ ]:
best_name = results_eval.loc[results_eval["RMSE"].idxmin(), "Model"]
best_model = best_models[best_name]
print(f"Melhor modelo: {best_name}")
best_preds = best_model.predict(X_valid)

fig = px.scatter(x=y_valid, y=best_preds, labels={"x": "Preço real", "y": "Preço previsto"}, title=f"Real vs Previsto ({best_name})")
fig.add_shape(type="line", x0=y_valid.min(), y0=y_valid.min(), x1=y_valid.max(), y1=y_valid.max(), line=dict(color="red", dash="dash"))
fig2 = px.histogram(y_valid - best_preds, nbins=40, labels={"value": "Resíduo"}, title=f"Distribuição dos resíduos ({best_name})")

fig.show()
fig2.show()


## 11. Importância de variáveis

In [ ]:
if best_name in ["Random Forest", "XGBoost"]:
    importance = best_model.feature_importances_
    try:
        num_features = numeric_cols
        cat_features = preprocessor.named_transformers_["cat"]["encoder"].get_feature_names_out(categorical_cols)
        feature_names = list(num_features) + cat_features.tolist()
    except Exception:
        feature_names = [f"f_{i}" for i in range(len(importance))]
    imp_df = pd.DataFrame({"feature": feature_names, "importance": importance})
    imp_df = imp_df.sort_values("importance", ascending=False).head(20)
    display(imp_df)
    fig3 = px.bar(imp_df, x="importance", y="feature", orientation="h", title="Top 20 importâncias de variáveis")
    fig3.show()
else:
    print("O KNN não oferece importância de variável de forma nativa.")


## 12. Dashboard interativo com Streamlit

A seguir, há um exemplo de app Streamlit que cria um dashboard interativo com os mesmos dados e os mesmos modelos. Para rodar o dashboard, execute na pasta do projeto:

In [ ]:
# No terminal, execute:
# streamlit run streamlit_dashboard.py


O dashboard permite ajustar hiperparâmetros em tempo real e ver gráficos de desempenho automaticamente.